## Reproducing ResNet on CIFAR-10: Experiments on Network Depth, Batch Size, and Pooling

---
- Baseline Code Link: https://github.com/kuangliu/pytorch-cifar

In [1]:
'''Train CIFAR10 with PyTorch.'''
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

from torchsummary import summary

import os
import argparse

import matplotlib.pyplot as plt
import numpy as np

# from resnet_20_32_44_56_v1 import *
from resnet_20_32_44_56_v2 import *
from utils import progress_bar

In [2]:
parser = argparse.ArgumentParser(description='PyTorch CIFAR10 Training')
parser.add_argument('--lr', default=0.1, type=float, help='learning rate')
parser.add_argument('--resume', '-r', action='store_true',
                    help='resume from checkpoint')
# args = parser.parse_args()
args, _ = parser.parse_known_args()

# device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")
best_acc = 0   # best test accuracy
start_epoch = 0   # start from epoch 0 or last checkpoint epoch

device: mps


In [3]:
# Data
print('==> Preparing data..')
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=100, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

==> Preparing data..
Files already downloaded and verified
Files already downloaded and verified


In [5]:
# Model
print('==> Building Model..\n')

# net = ResNet20()
# net = ResNet32()
# net = ResNet44()
net = ResNet56()

# Check layer(type), output shape, param #
print('==> Model Summary')
summary(net, (3, 32, 32))

net = net.to(device)

if args.resume:
    # Load checkpoint.
    print('==> Resuming from checkpoint..')
    assert os.path.isdir('checkpoint'), 'Error: no checkpoint directory found!'
    checkpoint = torch.load('./checkpoint/ckpt.pth')
    net.load_state_dict(checkpoint['net'])
    best_acc = checkpoint['acc']
    start_epoch = checkpoint['epoch']

criterion = nn.CrossEntropyLoss()
# 'We use a weight decay of 0.0001 and momentum of 0.9' (p.7)
optimizer = optim.SGD(net.parameters(), lr=args.lr,
                      momentum=0.9, weight_decay=0.0001)
# 'We start with a learning rate of 0.1, divide it by 10 at 32k and 48k iterations, and terminate training at 64k iterations' (p.7)
# This code terminates training at the 200 epoch, so the learning rate is divided by 10 at the 100 and 150 epochs to match the same ratio as in the paper
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[100, 150], gamma=0.1)

==> Building Model..

==> Model Summary
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 32, 32]             432
       BatchNorm2d-2           [-1, 16, 32, 32]              32
            Conv2d-3           [-1, 16, 32, 32]           2,304
       BatchNorm2d-4           [-1, 16, 32, 32]              32
            Conv2d-5           [-1, 16, 32, 32]           2,304
       BatchNorm2d-6           [-1, 16, 32, 32]              32
        BasicBlock-7           [-1, 16, 32, 32]               0
            Conv2d-8           [-1, 16, 32, 32]           2,304
       BatchNorm2d-9           [-1, 16, 32, 32]              32
           Conv2d-10           [-1, 16, 32, 32]           2,304
      BatchNorm2d-11           [-1, 16, 32, 32]              32
       BasicBlock-12           [-1, 16, 32, 32]               0
           Conv2d-13           [-1, 16, 32, 32]           2,304

In [6]:
# Training
def train(epoch):
    print('\nEpoch: %d' % epoch)
    net.train()
    train_loss = 0
    correct = 0
    total = 0
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        progress_bar(batch_idx, len(trainloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d))'
                     % (train_loss/(batch_idx+1), 100.*correct/total, correct, total))


def test(epoch):
    global best_acc
    net.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            progress_bar(batch_idx, len(testloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d))'
                         % (test_loss/(batch_idx+1), 100.*correct/total, correct, total))

    # Save checkpoint.
    acc = 100.*correct/total
    if acc > best_acc:
        print('Saving..')
        state = {
            'net': net.state_dict(),
            'acc': acc,
            'epoch': epoch,
        }
        if not os.path.isdir('checkpoint'):
            os.mkdir('checkpoint')
        torch.save(state, './checkpoint/ckpt.pth')
        best_acc = acc

In [7]:
for epoch in range(start_epoch, start_epoch+200):
    train(epoch)
    test(epoch)
    scheduler.step()


Epoch: 0
  Step: 303ms | Tot: 40s165ms | Loss: 2.222 | Acc: 19.934% (9967/50000) 391/391  60/391 
  Step: 23ms | Tot: 2s182ms | Loss: 1.769 | Acc: 32.890% (3289/10000) 100/100 
Saving..

Epoch: 1
  Step: 102ms | Tot: 39s719ms | Loss: 1.657 | Acc: 38.212% (19106/50000) 391/391 271/391 296/391 358/391 375/391 
  Step: 20ms | Tot: 2s141ms | Loss: 1.739 | Acc: 39.050% (3905/10000) 100/100 100 
Saving..

Epoch: 2
  Step: 101ms | Tot: 39s638ms | Loss: 1.406 | Acc: 48.734% (24367/50000) 391/391 249/391 
  Step: 21ms | Tot: 2s151ms | Loss: 1.538 | Acc: 47.460% (4746/10000) 100/100 
Saving..

Epoch: 3
  Step: 103ms | Tot: 39s781ms | Loss: 1.208 | Acc: 56.326% (28163/50000) 391/391 391 100/39 228/391 337/391 
  Step: 21ms | Tot: 2s156ms | Loss: 1.244 | Acc: 56.430% (5643/10000) 100/100 
Saving..

Epoch: 4
  Step: 102ms | Tot: 39s914ms | Loss: 1.039 | Acc: 63.192% (31596/50000) 391/391 45/391 174/391 291/391 
  Step: 21ms | Tot: 2s172ms | Loss: 1.097 | Acc: 62.460% (6246/10000) 100/100 
Saving..

  Step: 102ms | Tot: 39s872ms | Loss: 0.195 | Acc: 93.126% (46563/50000) 391/391 9 40/391 359/391 
  Step: 21ms | Tot: 2s140ms | Loss: 0.531 | Acc: 84.090% (8409/10000) 100/100 /100 

Epoch: 81
  Step: 102ms | Tot: 39s959ms | Loss: 0.191 | Acc: 93.224% (46612/50000) 391/391 1 127/391 
  Step: 21ms | Tot: 2s172ms | Loss: 0.427 | Acc: 87.050% (8705/10000) 100/100 00 

Epoch: 82
  Step: 101ms | Tot: 40s9ms | Loss: 0.197 | Acc: 92.988% (46494/50000) 391/391 1 16/391 
  Step: 21ms | Tot: 2s174ms | Loss: 0.445 | Acc: 86.800% (8680/10000) 100/100 0 

Epoch: 83
  Step: 103ms | Tot: 40s79ms | Loss: 0.192 | Acc: 93.222% (46611/50000) 391/391  
  Step: 20ms | Tot: 2s132ms | Loss: 0.561 | Acc: 83.840% (8384/10000) 100/100 

Epoch: 84
  Step: 104ms | Tot: 40s192ms | Loss: 0.192 | Acc: 93.176% (46588/50000) 391/391 /391 
  Step: 22ms | Tot: 2s185ms | Loss: 0.436 | Acc: 86.680% (8668/10000) 100/100 

Epoch: 85
  Step: 101ms | Tot: 40s238ms | Loss: 0.191 | Acc: 93.282% (46641/50000) 391/391 
  Step: 2

  Step: 22ms | Tot: 2s168ms | Loss: 0.364 | Acc: 91.970% (9197/10000) 100/100 100 

Epoch: 160
  Step: 106ms | Tot: 39s987ms | Loss: 0.006 | Acc: 99.894% (49947/50000) 391/391 119/391 259/391 284/391 377/391 
  Step: 21ms | Tot: 2s154ms | Loss: 0.364 | Acc: 92.070% (9207/10000) 100/100 

Epoch: 161
  Step: 116ms | Tot: 40s82ms | Loss: 0.006 | Acc: 99.880% (49940/50000) 391/391   216/39 271/391 
  Step: 23ms | Tot: 2s169ms | Loss: 0.362 | Acc: 91.970% (9197/10000) 100/100 100 

Epoch: 162
  Step: 104ms | Tot: 40s140ms | Loss: 0.006 | Acc: 99.878% (49939/50000) 391/391 
  Step: 23ms | Tot: 2s147ms | Loss: 0.364 | Acc: 91.990% (9199/10000) 100/100 

Epoch: 163
  Step: 102ms | Tot: 40s281ms | Loss: 0.006 | Acc: 99.888% (49944/50000) 391/391 
  Step: 21ms | Tot: 2s142ms | Loss: 0.364 | Acc: 91.920% (9192/10000) 100/100 

Epoch: 164
  Step: 102ms | Tot: 39s823ms | Loss: 0.006 | Acc: 99.884% (49942/50000) 391/391 91 84/39 259/39 272/39 329/39 360/391 
  Step: 21ms | Tot: 2s150ms | Loss: 0.363

In [8]:
print('Accuracy:', round(best_acc, 2))
print('Error:', round(100-best_acc, 2))

Accuracy: 92.15
Error: 7.85
